# 추론 시간 측정

In [ ]:
"""
Midm-2.0-Mini-Instruct (LoRA 병합 모델) 추론 + 시간 측정 스크립트

- ./merged_midm_epoch3 디렉터리 기준
- token_type_ids 제거 (LLaMA 계열)
- <DESC> / <JSON> 블록 분리
- N번 반복 추론 후 평균 추론 시간 출력
"""

import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ====================================
# 1. 경로 및 기본 설정
# ====================================

MODEL_DIR = "./merged_midm_epoch3"  # LoRA merge 결과 디렉터리

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SYSTEM_PROMPT = (
    "You are an AI assistant specialized in interior design description and image prompt generation. "
    "Given an input condition from the user, you must reproduce the exact style and structure of the target outputs "
    "shown in the training dataset, including both the natural language description and the JSON block for image models. "
    "Do not add explanations. Do not change the output format. "
    "Always follow the patterns demonstrated in the dataset."
)

GEN_KWARGS = {
    "max_new_tokens": 512,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "repetition_penalty": 1.18,
}


# ====================================
# 2. 프롬프트 빌더 / 후처리 함수
# ====================================

def build_inference_prompt(user_input: str) -> str:
    """
    학습 시와 동일한 구조의 프롬프트 생성
    """
    return (
        f"<s>[SYSTEM]\n{SYSTEM_PROMPT}\n\n"
        f"[USER]\n{user_input}\n\n"
        f"[ASSISTANT]\n"
    )


def split_desc_json(text: str):
    """
    모델 출력에서 <DESC> / <JSON> 블록 분리
    """
    desc = None
    js = None

    start_d = text.find("<DESC>")
    end_d = text.find("</DESC>")
    if start_d != -1 and end_d != -1:
        desc = text[start_d + len("<DESC>"): end_d].strip()

    start_j = text.find("<JSON>")
    end_j = text.find("</JSON>")
    if start_j != -1 and end_j != -1:
        js = text[start_j + len("<JSON>"): end_j].strip()

    return desc, js


# ====================================
# 3. 메인: 모델 로드 + 예시 추론 + 시간 측정
# ====================================

def main():
    # ----- 3-1. 모델 / 토크나이저 로드 -----
    print(f"[INFO] Loading merged model from: {MODEL_DIR}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=False)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        device_map="auto",
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        trust_remote_code=True,
    )
    model.eval()

    # ----- 3-2. 사용자 입력 예시 -----
    user_input = "화이트와 우드 조합의 따뜻한 거실, 간접조명과 소파, TV장이 어울리는 편안한 공간"

    print("\n===== 사용자 입력 =====")
    print(user_input)

    # 미리 토크나이즈 (프롬프트는 매번 동일하다고 가정)
    prompt = build_inference_prompt(user_input)
    base_inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    )
    base_inputs.pop("token_type_ids", None)  # LLaMA 계열: token_type_ids 제거
    base_inputs = {k: v.to(model.device) for k, v in base_inputs.items()}

    # ----- 3-3. 1회 추론: 생성 결과 확인 -----
    with torch.no_grad():
        gen_ids = model.generate(
            **base_inputs,
            eos_token_id=tokenizer.eos_token_id,
            **GEN_KWARGS,
        )

    full_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
    if "[ASSISTANT]" in full_text:
        full_text = full_text.split("[ASSISTANT]", 1)[1].strip()

    desc, js = split_desc_json(full_text)

    print("\n===== 전체 생성 결과 (1회 예시) =====")
    print(full_text)

    print("\n===== DESC 블록 (1회 예시) =====")
    print(desc if desc is not None else "(DESC 블록 없음)")

    print("\n===== JSON 블록 (1회 예시) =====")
    print(js if js is not None else "(JSON 블록 없음)")

    # ----- 3-4. N회 반복 추론 시간 측정 -----
    N = 20  # 반복 횟수 (원하면 10~50 사이로 조정 가능)
    times = []

    print(f"\n[INFO] 추론 시간 측정 시작 (N={N}) ...")

    for _ in range(N):
        # 동일 프롬프트 기준 반복 측정
        inputs = {k: v.clone() for k, v in base_inputs.items()}

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        with torch.no_grad():
            _ = model.generate(
                **inputs,
                eos_token_id=tokenizer.eos_token_id,
                **GEN_KWARGS,
            )

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        times.append(t1 - t0)

    avg_time = sum(times) / len(times)
    print(f"\n===== 평균 추론 시간 =====")
    print(f"단일 요청(512 토큰 생성 기준) 평균: {avg_time:.3f} 초/건 (N={N})")


if __name__ == "__main__":
    main()